<a href="https://colab.research.google.com/github/Anonymous-AV/DataScience-By-Aryan/blob/main/Titanic_%E2%80%93_Machine_Learning_from_Disaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCSE 1593 – Data Science and Visualization Lab

## Practical: Data Cleaning and Preparation

In this practical, we will take a raw dataset and prepare it for further analysis.

The work will follow a simple sequence:

> **Load → inspect → find problems → clean → transform → verify**

We will work with the **Titanic** dataset, a well-known dataset used in the Kaggle *Titanic – Machine Learning from Disaster* competition.

### In this practical, we will learn to

- identify missing values
- measure missing data column-wise
- decide when to remove rows or columns
- fill missing numeric values
- fill missing categorical values
- remove duplicate records
- convert columns to suitable data types
- create useful derived columns
- clean text values when text cleaning is part of preparation
- extract useful information from a string column
- verify the cleaned dataset

The focus is not on making the data look perfect. The focus is on making sensible decisions before analysis or modelling.

# Part A – Getting Started

## 1. Importing pandas

Pandas will be our main tool for working with the dataset.

### Live coding

Import pandas using the standard alias `pd`.

In [ ]:
print("Importing pandas")
import pandas as pd
# Write the import statement here


Importing pandas


## 2. Importing NumPy

NumPy is useful when we need `NaN`, numerical operations, or conditional transformations.

### Live coding

Import NumPy using the standard alias `np`.

In [ ]:
print("Importing NumPy")

# Write the import statement here


Importing NumPy


# Part B – Loading the Dataset

## 3. Dataset source – Titanic

The Titanic dataset is available through Kaggle's **Titanic – Machine Learning from Disaster** competition.

### Kaggle source

- Kaggle competition: https://www.kaggle.com/c/titanic
- Kaggle tutorial: https://www.kaggle.com/code/alexisbcook/titanic-tutorial

For this notebook, the CSV is loaded directly from a public raw-data URL. This avoids a manual download and keeps the practical reproducible.

### Direct CSV URL

```text
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
```

The file contains passenger information such as class, name, sex, age, ticket, fare, cabin and port of embarkation, along with the survival outcome.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

#read the csv data from this url into titanic
titanic=pd.read_csv(url)
#display first five rows
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 4. First look at the data

Before changing anything, it is useful to understand what we have.

Check:

- number of rows and columns
- column names
- data types
- first few records
- last few records

In [ ]:
print("Shape:")
print(titanic.shape)
print("\nColumns:")
print(titanic.columns)
print("\nData types:")
print(titanic.dtypes)

print("\nFirst five rows:")

titanic.head()
print("\nLast five rows:")
titanic.tail()


Shape:
(891, 12)

Columns:
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Columns:

Data types:

First five rows:

Last five rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


## 5. Basic summary

`describe()` gives a quick numerical summary. It can help us notice unusual values and get a feel for the scale of the data.

In [ ]:
#display numerical summary of dataset
titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# Part C – Finding Missing Data

## 6. What is a missing value?

A missing value means that a value is not available in the dataset.

Pandas usually represents missing numerical or general values as `NaN`.

Missing data matters because:

- calculations may ignore missing values
- some algorithms cannot work directly with missing values
- removing too many rows can reduce the dataset
- filling values without thinking can introduce bias

There is no single rule that works for every column.

## 7. Count missing values

`isna()` identifies missing values. Adding the result column-wise gives the number of missing values in each column.

### Live coding

Find the number of missing values in every column.

In [ ]:
#Calculate and display the number of missing values in each columns of dataset titanic
titanic.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


## 8. Missing-value percentage

A count is useful, but a percentage gives more context.

For a column:

	Missing % = (Missing values / Total rows) X  100

### Live coding

Create a table containing:

- missing count
- missing percentage

Sort it from highest to lowest missing percentage.

In [ ]:
#Create a table contaning missing count and missing percentage of titanic
titanic_missing = pd.DataFrame({
   'Missing Count': titanic.isna().sum(),
    'Missing Percentage': (titanic.isna().sum() / len(titanic)) * 100
})
#Sort it from highest to lowest percentage
titanic_missing = titanic_missing.sort_values(by='Missing Percentage', ascending=False)
#display it
titanic_missing

,Missing Count,Missing Percentage
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000


## 9. Which rows contain missing data?

Sometimes we need to inspect the actual records rather than only the totals.

### Live coding

Display the first few rows where `Age` is missing.

In [ ]:
#Display first few rows where age is missing
titanic[titanic['Age'].isna()].head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
17,18,1,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
26,27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
28,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q


## 10. Missing data by column

The Titanic data gives us three useful examples:

- `Age` has some missing values.
- `Cabin` has a large amount of missing data.
- `Embarked` has only a small number of missing values.

The appropriate action can therefore be different for different columns.

# Part D – Deciding What to Do with Missing Data

## 11. Option 1 – Remove rows

If only a very small number of records are incomplete and those records are not important for the analysis, removing them can be reasonable.

`dropna()` removes rows containing missing values.

### Example

Create a separate DataFrame containing only rows with a known `Embarked` value and compare length with titanic

In [ ]:
#No missing embarked dataset
titanic_embarked = titanic.dropna(subset=["Embarked"])
print(len(titanic_embarked))
#Original length
len(titanic)
print(len(titanic))
#length after removing missing
print(len(titanic)-len(titanic_embarked))

889
891
2


## 12. Option 2 – Fill missing numerical values

For a numerical column, common choices include:

- mean
- median
- a value based on domain knowledge

The **median** is often a useful choice when the distribution contains outliers.

For `Age`, we can inspect the median first.

In [ ]:
#calculate and display median age
int(titanic["Age"].median())# in int value convert
titanic["Age"].median()

28.0

## 13. Fill missing `Age` values with the median

We will create a working copy instead of modifying the original dataset immediately.

This makes it easier to compare the raw and cleaned versions.

In [ ]:
#create copy named titanic_clean
titanic_clean = titanic.copy()
print(titanic_clean)
#Fill median age
print("Fill median age")
titanic_clean["Age"] = titanic_clean["Age"].fillna(titanic_clean["Age"].median())
print(titanic_clean["Age"])
#check number of missing age values
titanic["Age"].isna().sum()


     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

np.int64(177)

## 14. Option 3 – Fill missing categorical values

For a categorical column, a common simple approach is to use the most frequent category, also called the **mode**.

For `Embarked`, first check the frequency of each category.

In [ ]:
titanic_clean["Embarked"].value_counts(dropna=False)
print(titanic_clean["Embarked"].value_counts(dropna=False))

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64


### Live coding

Find the mode of `Embarked` and use it to fill its missing values.

In [ ]:
#Find the mode of embarked
titanic_clean["Embarked"].mode()

# Fill the missing Embarked values

titanic_clean["Embarked"] = titanic_clean["Embarked"].fillna(titanic_clean["Embarked"].mode()[0])
print("Missing Embarked values:", titanic_clean["Embarked"].isna().sum())

Missing Embarked values: 0


## 15. Handling a column with a large amount of missing data

`Cabin` contains many missing values. Simply filling every missing cabin with the same cabin number would create information that was never present.

For this practical, we will keep a simpler approach: create a new indicator showing whether a cabin value was available, then remove the original `Cabin` column from the main cleaned table.

This preserves one useful piece of information without pretending that the missing cabin numbers are known.

In [ ]:
titanic_clean["CabinKnown"] = titanic_clean["Cabin"].notna().astype(int)

titanic_clean = titanic_clean.drop(columns=["Cabin"])

titanic_clean[["CabinKnown"]].value_counts()

KeyError: 'Cabin'

## 16. Checking the result of missing-data treatment

A good cleaning step should be followed by a check.

### Live coding

Display the remaining missing values.

In [ ]:
titanic_clean.isna().sum()

# Part E – Removing Duplicate Records

## 17. Duplicate rows

Duplicate records can affect counts, averages and later analysis.

Use `duplicated()` to identify duplicate rows.

In [ ]:
print("Number of duplicate rows:", titanic_clean.duplicated().sum())

## 18. Remove duplicate rows

`drop_duplicates()` returns a version without exact duplicate rows.

### Live coding

Remove duplicate rows from `titanic_clean` and check the new row count.

In [ ]:
before = len(titanic_clean)

#Clean duplicated rows from titanic_clean


after = len(titanic_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

# Part F – Data Transformation

## 19. What is data transformation?

Data transformation means changing data into a more useful form for analysis.

Examples include:

- converting data types
- creating new variables
- combining existing columns
- grouping continuous values into categories
- changing numerical scales

The transformation should have a clear purpose.

## 20. Check and convert data types

Pandas usually detects data types automatically, but it is good practice to check them.

For example, `Pclass` represents passenger class. Treating it as a numeric value is useful for some calculations, while treating it as a category can be useful for categorical analysis.

For this lab, we will keep the original `Pclass` numeric and create additional prepared columns where useful.

In [ ]:
titanic_clean.dtypes

## 21. Creating `FamilySize`

The dataset contains:

- `SibSp` – number of siblings or spouses aboard
- `Parch` – number of parents or children aboard

We can combine them with the passenger to create a simple family-size variable:

    FamilySize = SibSp + Parch + 1

The `1` represents the passenger themselves.

In [ ]:
#Create a new column familysize

#Display first five rows with SibSp, Parch & Familysize


## 22. Creating `IsAlone`

A passenger with a family size of 1 travelled alone.

This converts a numerical idea into a simple binary feature.

In [ ]:
#Create a column IsAlone for family size 1 as int type


titanic_clean[["FamilySize", "IsAlone"]].head(10)

## 23. Creating `FarePerPerson`

The same ticket may represent more than one passenger. A simple preparation step is to divide the fare by the family size.

This is only a derived feature for practice; it is not a statement about the actual ticket pricing rules.

In [ ]:
#A Columns Fare per person

titanic_clean[["Fare", "FamilySize", "FarePerPerson"]].head()

## 24. Grouping age into categories

Sometimes an exact numerical value is less convenient for a particular analysis.

We can create age groups such as:

- Child
- Teenager
- Adult
- Senior

### Live coding

Use `pd.cut()` to create an `AgeGroup` column.

In [ ]:
bins = [0, 12, 19, 59, np.inf]
labels = ["Child", "Teenager", "Adult", "Senior"]

titanic_clean["AgeGroup"] = pd.cut(
    titanic_clean["Age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

titanic_clean[["Age", "AgeGroup"]].head(15)

# Part G – String Cleaning and Preparation

## 25. Why string manipulation belongs here

Text columns often contain useful information, but they may need cleaning before they can be analysed.

In this dataset, `Name` contains a person's name and title in one string. Extracting the title creates a useful structured feature.

We will also use simple string operations such as:

- `str.strip()` to remove unwanted spaces
- `str.lower()` to standardise text
- `str.split()` to separate parts of a string
- `str.extract()` to pull a pattern from text

These operations are being used as part of data preparation, not as a separate string-programming exercise.

## 26. Remove extra spaces from `Name`

Some names contain trailing spaces. `str.strip()` removes spaces at the beginning and end of each value.

In [ ]:
#Remove extra spaces from name

titanic_clean["Name"].head()

## 27. Standardise `Sex`

Text categories are easier to work with when they follow a consistent format.

We will convert the values to lowercase and remove surrounding spaces.

In [ ]:
#Remove space and make sex as lowercase


titanic_clean["Sex"].value_counts(dropna=False)

## 28. Extracting a title from `Name`

The names follow a pattern similar to:

```text
Surname, Title. Given Name
```

For example:

```text
Braund, Mr. Owen Harris
```

The title is useful as a compact categorical variable.

### Live coding

Use `str.extract()` with a regular expression to create a `Title` column.

In [ ]:
titanic_clean["Title"] = titanic_clean["Name"].str.extract(r",\s*([^.]*)\.", expand=False)

titanic_clean["Title"].value_counts().head(15)

## 29. Clean the extracted titles

Some titles occur very rarely. For a simple preparation step, we can group uncommon titles into `Rare`.

First, look at the title counts.

In [ ]:
title_counts = titanic_clean["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index

print("Rare titles:")
print(list(rare_titles))

### Group uncommon titles

Replace titles occurring fewer than 10 times with `Rare`.

In [ ]:
titanic_clean["Title"] = titanic_clean["Title"].replace(rare_titles, "Rare")

titanic_clean["Title"].value_counts()

## 30. Clean `Embarked` labels

`Embarked` contains short categorical codes. We can standardise the text and optionally map the codes to readable names.

First, make sure the text is consistent.

In [ ]:
#embark to uppercase

titanic_clean["Embarked"].value_counts(dropna=False)

### Map codes to readable labels

For reporting, readable category names can be more convenient.

In [ ]:
port_map = {
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
}

#map embarked port codes with port names into a new columns embarkedname

titanic_clean[["Embarked", "EmbarkedName"]].head(10)

# Part H – A Small Cleaning Pipeline

## 31. Putting the main steps together

Once the individual operations are clear, cleaning can be written as a sequence of steps.

The following version starts again from the raw dataset so that the process is easy to reproduce.

In [ ]:
cleaned = titanic.copy()

# 1. Remove unnecessary spaces from text columns
cleaned["Name"] = cleaned["Name"].str.strip()
cleaned["Sex"] = cleaned["Sex"].str.strip().str.lower()
cleaned["Embarked"] = cleaned["Embarked"].str.strip().str.upper()

# 2. Fill numerical missing values
cleaned["Age"] = cleaned["Age"].fillna(cleaned["Age"].median())

# 3. Fill categorical missing values
cleaned["Embarked"] = cleaned["Embarked"].fillna(cleaned["Embarked"].mode()[0])

# 4. Preserve whether Cabin information was available
cleaned["CabinKnown"] = cleaned["Cabin"].notna().astype(int)
cleaned = cleaned.drop(columns=["Cabin"])

# 5. Remove exact duplicates
cleaned = cleaned.drop_duplicates().copy()

# 6. Create derived variables
cleaned["FamilySize"] = cleaned["SibSp"] + cleaned["Parch"] + 1
cleaned["IsAlone"] = (cleaned["FamilySize"] == 1).astype(int)
cleaned["FarePerPerson"] = cleaned["Fare"] / cleaned["FamilySize"]

# 7. Create age groups
cleaned["AgeGroup"] = pd.cut(
    cleaned["Age"],
    bins=[0, 12, 19, 59, np.inf],
    labels=["Child", "Teenager", "Adult", "Senior"],
    include_lowest=True
)

# 8. Extract and simplify title
cleaned["Title"] = cleaned["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
title_counts = cleaned["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index
cleaned["Title"] = cleaned["Title"].replace(rare_titles, "Rare")

cleaned.head()

## 32. Verify the cleaned dataset

Cleaning is not finished until we check the result.

Look at:

- shape
- data types
- missing values
- duplicate rows
- a few records

In [ ]:
print("Shape:", cleaned.shape)
print("\nData types:")
print(cleaned.dtypes)
print("\nMissing values:")
print(cleaned.isna().sum())
print("\nDuplicate rows:", cleaned.duplicated().sum())
print("\nSample records:")
display(cleaned.head())

## 33. Compare raw and cleaned data

It is useful to see what changed rather than assuming that the cleaning worked.

In [ ]:
cleaned = titanic.copy()

# Remove rows with missing values
cleaned = cleaned.dropna()

In [ ]:
comparison = pd.DataFrame({
    "Raw Rows": [len(titanic)],
    "Cleaned Rows": [len(cleaned)],
    "Raw Columns": [len(titanic.columns)],
    "Cleaned Columns": [len(cleaned.columns)],
    "Raw Missing Values": [titanic.isna().sum().sum()],
    "Cleaned Missing Values": [cleaned.isna().sum().sum()],
})

comparison

,Raw Rows,Cleaned Rows,Raw Columns,Cleaned Columns,Raw Missing Values,Cleaned Missing Values
0,891,183,12,12,866,0


# Part I – Save the Prepared Dataset

## 34. Save the cleaned data

A cleaned dataset is often saved so that the preparation work does not need to be repeated every time.

### Live coding

Save the prepared DataFrame as `titanic_cleaned.csv` and load it back to verify the file.

In [ ]:
output_file = "titanic_cleaned.csv"
cleaned.to_csv(output_file, index=False)

reloaded = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Reloaded shape:", reloaded.shape)
display(reloaded.head())

# Part J – Quick Checks

Before moving to analysis, ask these questions:

- Did I identify missing values before filling or removing them?
- Did I choose a reasonable treatment for each column?
- Did I avoid inventing values that were not known?
- Did I remove exact duplicates?
- Are numerical columns still numerical?
- Are categorical/text columns consistent?
- Do the new columns have a clear purpose?
- Did I check the dataset again after cleaning?

A clean dataset is not necessarily a dataset with zero missing values. Sometimes missingness itself contains useful information.

# Practice Set

The following exercises use the `cleaned` DataFrame unless another DataFrame is specified. Try to solve them without changing the original `titanic` DataFrame.

## Question 1 – Missing-value audit

Create a table showing the missing count and missing percentage for every column in `titanic`. Sort the result from highest to lowest missing percentage.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here

## Question 2 – Missing Age

Create a copy of `titanic` called `q2`. Fill missing `Age` values with the median age. Verify that no missing age values remain.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here

## Question 3 – Categorical missing data

In a new DataFrame, fill missing `Embarked` values using the mode. Display the rows that originally had missing `Embarked` and show their new values.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here

## Question 4 – Compare imputation methods

Create two copies of the Titanic data. Fill missing `Age` with the mean in one and the median in the other. Compare the two replacement values and write one or two observations as comments.

### Live coding
Write your solution in the cell below.

In [ ]:
# Create two copies of Titanic data
titanic_mean = titanic.copy()
titanic_median = titanic.copy()

# Calculate mean and median Age
mean_age = titanic["Age"].mean()
median_age = titanic["Age"].median()

# Fill missing Age values
titanic_mean["Age"] = titanic_mean["Age"].fillna(mean_age)
titanic_median["Age"] = titanic_median["Age"].fillna(median_age)

# Compare replacement values
print("Mean Age:", mean_age)
print("Median Age:", median_age)

# Observations:
# 1. Mean is affected by extremely high or low ages.
# 2. Median is less affected by outliers and can be more robust.

Mean Age: 29.69911764705882
Median Age: 28.0


## Question 5 – Duplicate check

Check whether the raw Titanic dataset contains exact duplicate rows. Then create a duplicate manually and use `drop_duplicates()` to remove it.

### Live coding
Write your solution in the cell below.

In [ ]:
# Check exact duplicate rows in raw dataset
print("Duplicate rows:", titanic.duplicated().sum())

# Create a duplicate manually
titanic_duplicate = pd.concat([titanic, titanic.iloc[[0]]], ignore_index=True)

# Remove duplicate rows
titanic_duplicate = titanic_duplicate.drop_duplicates()

# Verify after removing duplicates
print("Duplicates after removal:", titanic_duplicate.duplicated().sum())

Duplicate rows: 0
Duplicates after removal: 0


## Question 6 – Family categories

Using `FamilySize` in `cleaned`, create a new column called `FamilyType` with three categories: `Alone` for 1, `Small` for 2–4, and `Large` for more than 4.

### Live coding
Write your solution in the cell below.

In [ ]:
# Create FamilySize if it does not exist
cleaned["FamilySize"] = cleaned["SibSp"] + cleaned["Parch"] + 1

# Create FamilyType
cleaned["FamilyType"] = cleaned["FamilySize"].apply(
    lambda x: "Alone" if x == 1 else "Small" if x <= 4 else "Large"
)

# Display results
cleaned[["FamilySize", "FamilyType"]].head()

,FamilySize,FamilyType
1,2,Small
3,2,Small
6,1,Alone
10,3,Small
11,1,Alone


## Question 7 – Fare transformation

Create a column called `FareRounded` by rounding `Fare` to two decimal places. Display `Fare` and `FareRounded` together.

### Live coding
Write your solution in the cell below.

In [ ]:
# Create FareRounded column
titanic["FareRounded"] = titanic["Fare"].round(2)

# Display both columns
titanic[["Fare", "FareRounded"]].head()

,Fare,FareRounded
0,7.2500,7.25
1,71.2833,71.28
2,7.9250,7.92
3,53.1000,53.10
4,8.0500,8.05


## Question 8 – Text cleaning

Create a new column from `Name` that contains only the surname. Use string operations rather than manually entering names.

### Live coding
Write your solution in the cell below.

In [ ]:
# Extract surname from Name
titanic["Surname"] = titanic["Name"].str.split(",").str[0]

# Display results
titanic[["Name", "Surname"]].head()

,Name,Surname
0,"Braund, Mr. Owen Harris",Braund
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Cumings
2,"Heikkinen, Miss. Laina",Heikkinen
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Futrelle
4,"Allen, Mr. William Henry",Allen


## Question 9 – Title analysis

Display the number of passengers in each `Title` category after rare titles have been grouped. Sort the result in descending order.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here
# Display passenger count for each Title
titanic["Title"].value_counts().sort_values(ascending=False)

,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2
Major,2


## Question 10 – Age categories

Create a different `AgeBand` column using the groups: `Child` (0–12), `Young` (13–25), `Adult` (26–59), and `Senior` (60+).

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here
# Create AgeBand column
titanic["AgeBand"] = pd.cut(
    titanic["Age"],
    bins=[-1, 12, 25, 59, float("inf")],
    labels=["Child", "Young", "Adult", "Senior"]
)

# Display results
titanic[["Age", "AgeBand"]].head(10)

,Age,AgeBand
0,22.0,Young
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult
5,NaN,NaN
6,54.0,Adult
7,2.0,Child
8,27.0,Adult
9,14.0,Young


## Question 11 – Final cleaning check

Create a compact report containing the number of rows, number of columns, total missing values and duplicate rows in `cleaned`.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here
print(titanic.columns.tolist())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'IsAlone', 'AgeGroup', 'Title', 'EmbarkedName']


## Question 12 – Mini case study

Create `final_titanic` containing `Survived`, `Pclass`, `Sex`, `Age`, `Fare`, `FamilySize`, `IsAlone`, `AgeGroup`, `Title`, and `EmbarkedName`. Verify its shape, data types and missing values.

### Live coding
Write your solution in the cell below.

In [ ]:
# Create FamilySize
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1

# Create IsAlone
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

# Create AgeGroup
titanic["AgeGroup"] = pd.cut(
    titanic["Age"],
    bins=[0, 12, 18, 60, float("inf")],
    labels=["Child", "Teenager", "Adult", "Senior"]
)

# Create Title
titanic["Title"] = titanic["Name"].str.extract(r",\s*([^.]*)\.")

# Create EmbarkedName
titanic["EmbarkedName"] = titanic["Embarked"].map({
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
})

# Create final dataset
final_titanic = titanic[[
    "Survived", "Pclass", "Sex", "Age", "Fare",
    "FamilySize", "IsAlone", "AgeGroup", "Title", "EmbarkedName"
]].copy()

# Verify results
print("Shape:", final_titanic.shape)
print("\nData Types:\n", final_titanic.dtypes)
print("\nMissing Values:\n", final_titanic.isnull().sum())

Shape: (891, 10)

Data Types:
 Survived           int64
Pclass             int64
Sex               object
Age              float64
Fare             float64
FamilySize         int64
IsAlone            int64
AgeGroup        category
Title             object
EmbarkedName      object
dtype: object

Missing Values:
 Survived          0
Pclass            0
Sex               0
Age             177
Fare              0
FamilySize        0
IsAlone           0
AgeGroup        177
Title             0
EmbarkedName      2
dtype: int64


# Final Challenge – Prepare a Dataset for Analysis

Imagine that another student will use your prepared data tomorrow without seeing the raw file.

Create a final DataFrame that:

1. contains no exact duplicate rows
2. has a sensible treatment for missing `Age` and `Embarked`
3. does not keep the mostly-missing `Cabin` column as raw text
4. contains a `CabinKnown` indicator
5. has cleaned text values in `Sex` and `Embarked`
6. contains `FamilySize` and `IsAlone`
7. contains `AgeGroup`
8. contains a cleaned `Title`
9. contains `FarePerPerson`
10. passes a final missing-value and data-type check

### Suggested final check

Display the final DataFrame's shape, columns, data types, missing-value counts and first five rows.

In [ ]:
# Build your final prepared DataFrame here.
# Use the steps from the practical and add your own checks.
# Build final prepared DataFrame

import pandas as pd

# Create new features
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

titanic["AgeGroup"] = pd.cut(
    titanic["Age"],
    bins=[0, 12, 18, 60, float("inf")],
    labels=["Child", "Teenager", "Adult", "Senior"]
)

titanic["Title"] = titanic["Name"].str.extract(r",\s*([^.]*)\.")

titanic["EmbarkedName"] = titanic["Embarked"].map({
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
})

# Select required columns
final_titanic = titanic[[
    "Survived", "Pclass", "Sex", "Age", "Fare",
    "FamilySize", "IsAlone", "AgeGroup", "Title", "EmbarkedName"
]].copy()

# Verification checks
print("Shape:", final_titanic.shape)
print("\nData Types:\n", final_titanic.dtypes)
print("\nMissing Values:\n", final_titanic.isnull().sum())
print("\nDuplicate Rows:", final_titanic.duplicated().sum())

# Display first five rows
final_titanic.head()

Shape: (891, 10)

Data Types:
 Survived           int64
Pclass             int64
Sex               object
Age              float64
Fare             float64
FamilySize         int64
IsAlone            int64
AgeGroup        category
Title             object
EmbarkedName      object
dtype: object

Missing Values:
 Survived          0
Pclass            0
Sex               0
Age             177
Fare              0
FamilySize        0
IsAlone           0
AgeGroup        177
Title             0
EmbarkedName      2
dtype: int64

Duplicate Rows: 104


,Survived,Pclass,Sex,Age,Fare,FamilySize,IsAlone,AgeGroup,Title,EmbarkedName
0,0,3,male,22.0,7.2500,2,0,Adult,Mr,Southampton
1,1,1,female,38.0,71.2833,2,0,Adult,Mrs,Cherbourg
2,1,3,female,26.0,7.9250,1,1,Adult,Miss,Southampton
3,1,1,female,35.0,53.1000,2,0,Adult,Mrs,Southampton
4,0,3,male,35.0,8.0500,1,1,Adult,Mr,Southampton


# Quick Reference

### Missing data

```python
df.isna().sum()
df.dropna()
df.dropna(subset=["Column"])
df["Column"].fillna(value)
df["Column"].fillna(df["Column"].median())
df["Column"].fillna(df["Column"].mode()[0])
```

### Duplicates

```python
df.duplicated().sum()
df.drop_duplicates()
```

### Transformation

```python
df["NewColumn"] = df["A"] + df["B"]
pd.cut(df["Age"], bins=[...], labels=[...])
```

### String cleaning

```python
df["Column"].str.strip()
df["Column"].str.lower()
df["Column"].str.upper()
df["Column"].str.extract(pattern)
```

### Final inspection

```python
df.shape
df.columns
df.dtypes
df.isna().sum()
df.head()
```

> **Good preparation is a cycle: inspect → decide → transform → verify.**